# NB01 – Daten Laden

**Grid-Arbitrage** · Batteriespeicher-Arbitrage im Schweizer Strommarkt

**Gruppe:** SC26_Gruppe_2 | **Verantwortlich:** Patrik Neunteufel | **Datum:** März 2026

---

*Lädt [ENTSO-E](../organisation/O_02_Glossar.ipynb#g-entsoe) Spot-Preise und CH-Netzlast von externen APIs in `data/raw/`.*


|  [← NB00 Business Case](00_Business_Case.ipynb) | [↑ Übersicht ↑](../organisation/O_01_Project_Overview.ipynb) | [NB02 Daten Bereinigung →](02_Daten_Bereinigung.ipynb) |
|:---|:---:|---:|


## Inhaltsverzeichnis<a id='toc_NB_01'></a>

1. [Einleitung](#einleitung_NB_01)
2. [Initialisierung](#initialisierung_NB_01)
3. [Datensatz 1: ENTSO-E Day-Ahead Spot-Preise CH](#datensatz-1-entso-e-day-ahead-spot-preise-ch_NB_01)
4. [Datensatz 2: CH Netzlast (Systemlast)](#datensatz-2-ch-netzlast-systemlast_NB_01)
5. [Fazit](#fazit_NB_01)
6. [Abschluss](#abschluss_NB_01)

---
## Einleitung <a id='einleitung_NB_01'></a>

[↑ Inhaltsverzeichnis](#toc_NB_01)

Laden der Pflicht-Rohdaten via ENTSO-E Transparency Platform:

- **Datensatz 1** — Day-Ahead-Spot-Preise CH (stündlich, EUR/MWh)
- **Datensatz 2** — Systemlast CH (stündlich, MW)

Download erfolgt jahresweise mit Retry-Logik, um API-Rate-Limits und temporäre
503-Fehler zu überbrücken. Output in `data/raw/` als CSV. Der tatsächlich
geladene Datenzeitraum wird nach `../sync/transfer.json` geschrieben, damit
NB02–K_99 damit arbeiten können.


## Initialisierung<a id='initialisierung_NB_01'></a>

[↑ Inhaltsverzeichnis](#toc_NB_01)

**Abhängigkeiten prüfen:** Fehlende Pakete (`pandas`, `numpy`, `requests`, `entsoe-py`)
werden automatisch per `pip` installiert, falls nicht vorhanden.


In [ ]:
# ── lib/ aus Projekt-Root erreichbar machen + lib-Imports ───────────────────
# Notebook liegt in einem Unterordner (kuer/, experimental/, notebooks/,
# organisation/). Damit 'from lib.xxx import ...' funktioniert, muss der
# Projekt-Root vorne in sys.path stehen. autoreload sorgt dafür, dass
# Änderungen in lib/*.py ohne Kernel-Restart übernommen werden.
import sys, os
_PROJECT_ROOT = os.path.abspath('..')
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

# lib-Imports (einmal zentral — in allen folgenden Zellen verfügbar)
from lib.plotting import show_source
from lib.data_fetchers  import fetch_entsoe_yearly
from lib.io_ops   import load_transfer, save_transfer, log_dataindex, needs_download, log_missing

print(f'lib-Pfad aktiv: {_PROJECT_ROOT}/lib')


In [ ]:
# ── Check, if used Libraries need to be installed ───────────────────────────
import subprocess, sys
for imp, pkg in [('pandas','pandas'),('requests','requests'),('numpy','numpy'),
                 ('entsoe','entsoe-py')]:
    try: __import__(imp)
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])


**Imports**
  
Import needed libraries


In [ ]:
import os, sys, warnings, json
import numpy  as np
import pandas as pd
import requests
from datetime import datetime
warnings.filterwarnings('ignore')

# Versionen anzeigen für Reproduzierbarkeit
# print(f"Os           Version: {os.__version__}")
# print(f"Sys          Version: {sys.__version__}")
# print(f"Warnings     Version: {warnings.__version__}")
print(f"Numpy        Version: {np.__version__}")
print(f"Pandas       Version: {pd.__version__}")
print(f"Requests     Version: {requests.__version__}")
print(f"📅 Zuletzt ausgeführt am: {datetime.now().strftime('%d.%m.%Y um %H:%M:%S')}")

**Setup – Konfiguration & Verzeichnisstruktur:** 

Lädt `../sync/config.json` (SSOT), setzt alle
Pfade, legt fehlende Verzeichnisse an und definiert `needs_download()` als Reload-Guard.


In [ ]:
# ── Projektstruktur & Konfiguration ──────────────────────────────────────────────────
# ── ../sync/config.json laden (Single Source of Truth) ───────────────────────────────
# Schalter NIE direkt hier setzen — immer in ../sync/config.json anpassen.
_cfg_path = '../sync/config.json'
if not os.path.exists(_cfg_path):
    raise FileNotFoundError(
        '../sync/config.json nicht gefunden. '
        'Bitte ins Projektverzeichnis legen (siehe NB00b Sektion 5).')
with open(_cfg_path) as _f:
    CFG = json.load(_f)

# ── Aliases (nur lesend — Quelle bleibt ../sync/config.json) ─────────────────────────
MODE         = CFG['mode']            # 'data' | 'sim'
FORCE_RELOAD = CFG['force_reload']    # dict mit allen Keys

# ── Datenzeitraum ────────────────────────────────────────────────────────────
_start   = CFG['daten']['start_year']
_end     = CFG['daten']['end_year']
START_YEAR = int(_start)
END_YEAR   = datetime.now().year if str(_end) == 'heute' else int(_end)
YEARS      = list(range(START_YEAR, END_YEAR + 1))
print(f'../sync/config.json geladen | MODE={MODE}')
print(f'Datenzeitraum: {START_YEAR}–{END_YEAR} | Jahre: {YEARS}')

# ── Verzeichnisstruktur ───────────────────────────────────────────────────────
DATA_FOLDER      = '../data'
OUTPUT_FOLDER    = '../output'
DIR_RAW          = os.path.join(DATA_FOLDER, 'raw')
DIR_PROCESSED    = os.path.join(DATA_FOLDER, 'processed')
DIR_INTERMEDIATE = os.path.join(DATA_FOLDER, 'intermediate')
DATAINDEX        = '../sync/dataindex.csv'
SZ_AKTIV         = CFG['szenarien']['gleichzeitigkeit_aktiv']  # SSOT

# Nur benötigte Ordner anlegen:
# - sim/ wird nur von NB01b angelegt (SIM-Mode)
# - output/kuer/ existiert nicht mehr in der aktuellen Architektur
# - Szenario-Unterordner: nur den aktiven anlegen (SZ_AKTIV aus config)
for d in [DIR_RAW, DIR_PROCESSED, DIR_INTERMEDIATE,
          os.path.join(DATA_FOLDER,'intermediate', SZ_AKTIV),   # nur aktiver Szenario-Ordner
          os.path.join(OUTPUT_FOLDER,'charts', SZ_AKTIV)]:       # Charts-Zielordner
    os.makedirs(d, exist_ok=True)

print(f'raw          : {os.path.abspath(DIR_RAW)}')
print(f'processed    : {os.path.abspath(DIR_PROCESSED)}')
print(f'intermediate : {os.path.abspath(DIR_INTERMEDIATE)}')

**Hilfsfunktionen:** `log_dataindex()` schreibt jede erzeugte Datei ins Datenprotokoll;
`log_missing()` markiert fehlgeschlagene Downloads und schreibt `data/missing.txt`.


**🔎 Quellcode der importierten lib-Funktion**

Die Funktion `log_dataindex` wird aus `lib/io_ops.py` importiert und
schreibt Einträge ins Daten-Provenienz-Protokoll `sync/dataindex.csv`.
Bereits bestehende Einträge für denselben Dateinamen werden als
`superseded` markiert. Aufklappbar darunter ist der Quellcode einsehbar.


In [ ]:
show_source(log_dataindex)


In [ ]:
# ── ../sync/dataindex.csv Helper ───────────────────────────────────────────────────────
print('dataindex Helper bereit.')


---
## Datensatz 1: ENTSO-E Day-Ahead Spot-Preise CH <a id='datensatz-1-entso-e-day-ahead-spot-preise-ch_NB_01'></a>

[↑ Inhaltsverzeichnis](#toc_NB_01)
    
**Quelle:** [ENTSO-E](../organisation/O_02_Glossar.ipynb#g-entsoe) Transparency Platform — `transparency.entsoe.eu`  
**Methode:** REST-[API](../organisation/O_02_Glossar.ipynb#g-api) via `entsoe-py` Python-Bibliothek (`query_day_ahead_prices`)  
**Zone:** `10YCH-SWISSGRIDZ` (Schweizer Bietungszone)  
**Format:** Stündliche Preise [EUR/MWh], [UTC](../organisation/O_02_Glossar.ipynb#g-utc)-Index  
**Zweck:** Hauptdatensatz für [Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch)-Simulation und Wirtschaftlichkeitsrechnung.  
Der Intra-Tag-Spread (p75 − p25) ist der direkte Erlöstreiber der [Grid-Arbitrage](../organisation/O_02_Glossar.ipynb#g-grid-arbitrage).


**🔎 Quellcode der importierten lib-Funktion**

Die Funktion `fetch_entsoe_yearly` wird aus `lib/data_fetchers.py` importiert und
kapselt die jahresweise ENTSO-E-Abfrage mit 503-Retry. Die konkrete
Query (Preise, Last, Grenzflüsse, …) wird als Lambda übergeben —
so bleibt die Retry-Logik unverändert für neue Queries. Aufklappbar
ist der Quellcode einsehbar.


In [ ]:
show_source(fetch_entsoe_yearly)


**🔎 Quellcode der importierten lib-Funktion**

`needs_download` aus `lib.io_ops` — aufklappbar ist der Quellcode einsehbar.


In [ ]:
show_source(needs_download)


In [ ]:
# API-Key aus ../sync/config.json (api_keys.entsoe) — nie im Code hardcoden
ENTSOE_API_KEY = (CFG.get('api_keys', {}).get('entsoe', '')
                  or os.environ.get('ENTSOE_API_KEY', ''))

PRICES_FILE = os.path.join(DIR_RAW, 'ch_spot_prices_raw.csv')

# ── Connectivity-Check (web-api Endpunkt, nicht nur Portal-Startseite) ───────
# Hinweis: transparency.entsoe.eu (HTTP 200) ≠ web-api.tp.entsoe.eu (kann 503 sein)
# Daher direkt den API-Endpunkt pingen statt die Webseite.
try:
    r = requests.get(
        'https://web-api.tp.entsoe.eu/api',
        params={'securityToken': ENTSOE_API_KEY, 'documentType': 'A44'},
        timeout=10)
    # 400 = Endpunkt erreichbar (Pflichtparameter fehlen — erwartetes Verhalten)
    # 401 = Endpunkt erreichbar, Key ungültig
    # 503 = Server temporär überlastet — ERREICHBAR, Retry übernimmt
    # → portal_ok = True für alle Codes ausser Netzwerkfehler
    portal_ok = r.status_code != 0
    print(f'ENTSO-E API-Endpunkt: HTTP {r.status_code} | '
          f'Key vorhanden: {bool(ENTSOE_API_KEY)}')
    if r.status_code == 503:
        print('  → 503: Server überlastet, aber erreichbar — Retry-Logik übernimmt.')
    elif r.status_code == 401:
        print('  → 401: API-Key ungültig — bitte prüfen.')
        portal_ok = False
except Exception as e:
    portal_ok = False
    print(f'ENTSO-E nicht erreichbar (Netzwerk?): {e}')


if not needs_download(PRICES_FILE, 10, 'spot_prices'):
    print(f'Preisdaten vorhanden ({os.path.getsize(PRICES_FILE)/1024:.0f} KB) – kein Re-Download.')
    df_prices = pd.read_csv(PRICES_FILE)

elif ENTSOE_API_KEY and portal_ok:
    print(f'Lade ENTSO-E CH Preise {START_YEAR}–{END_YEAR} ({len(YEARS)} Jahre, max. 3 Retries bei 503)...')
    from entsoe import EntsoePandasClient
    client = EntsoePandasClient(api_key=ENTSOE_API_KEY)
    frames = []
    for year in YEARS:
        print(f'  Lade {year}...', end=' ')
        ts_year = fetch_entsoe_yearly(
                lambda s, e: client.query_day_ahead_prices('CH', start=s, end=e),
                year)
        frames.append(ts_year)
        print(f'{len(ts_year):,} h OK')
    ts = pd.concat(frames).sort_index()
    ts = ts[~ts.index.duplicated(keep='first')]  # Überlapp-Schutz
    df_prices = ts.rename('price_eur_mwh').to_frame().reset_index()
    df_prices.columns = ['timestamp', 'price_eur_mwh']
    df_prices['timestamp'] = pd.to_datetime(df_prices['timestamp'], utc=True)
    df_prices.to_csv(PRICES_FILE, index=False, encoding='utf-8')
    kb = os.path.getsize(PRICES_FILE) / 1024
    log_dataindex('ch_spot_prices_raw.csv',
                  'https://transparency.entsoe.eu', PRICES_FILE, 'raw',
                  rows=len(df_prices), size_kb=kb, note=f'ENTSO-E API, entsoe-py, {START_YEAR}-{END_YEAR}')
    print(f'Gespeichert: {PRICES_FILE} | {len(df_prices):,} h | {kb:.0f} KB')
else:
    reason = 'Kein API-Key' if not ENTSOE_API_KEY else 'Portal/API nicht erreichbar (503?)'
    log_missing('ch_spot_prices_raw.csv', reason)
    raise RuntimeError(
        f'{reason}\n'
        '→ Bei 503: In 15-30 Min erneut versuchen (ENTSO-E hat gelegentlich Wartungsfenster).\n'
        '→ API-Key beantragen: transparency@entsoe.eu, Betreff: Restful API access\n'
        '→ Oder sofort weiterarbeiten: NB1b_Daten_Sim ausführen.')

print(f'Qualitaet: {len(df_prices):,} Zeilen | '
      f'Min: {df_prices["price_eur_mwh"].min():.1f} / '
      f'Max: {df_prices["price_eur_mwh"].max():.1f} EUR/MWh')


**Verifikation DS1:** Shape, Zeitraum und Wertebereich der Spot-Preise prüfen.


In [ ]:
# ── Verifikation DS1: Spot-Preise ───────────────────────────────────────────
print(f'Shape   : {df_prices.shape}')
print(f'Zeitraum: {df_prices["timestamp"].min()} – {df_prices["timestamp"].max()}')
print(f'Nulls   : {df_prices.isnull().sum().sum()}')
print(f'Range   : {df_prices["price_eur_mwh"].min():.1f} – '
      f'{df_prices["price_eur_mwh"].max():.1f} EUR/MWh')
print(f"Pandas Version: {pd.__version__}")
df_prices.head(3)

---
## Datensatz 2: CH Netzlast (Systemlast) <a id='datensatz-2-ch-netzlast-systemlast_NB_01'></a>

[↑ Inhaltsverzeichnis](#toc_NB_01)
    
**Quelle:** [ENTSO-E](../organisation/O_02_Glossar.ipynb#g-entsoe) Transparency Platform — `transparency.entsoe.eu`  
**Methode:** REST-[API](../organisation/O_02_Glossar.ipynb#g-api) via `entsoe-py` (`query_load('CH')`)  
**Identisch mit DS1:** Gleicher API-Key und Endpunkt — eine Moodle-URL für DS1+DS2.  
**Format:** Stündliche Systemlast [MW→GW], [UTC](../organisation/O_02_Glossar.ipynb#g-utc)-Index  
**Zweck:** Netzentlastungsmodell — aggregierte Batterien sollen die Spitzenlast senken.  
Koinzidenz mit Preishochpunkten bestätigt Business Case (Import teuer = Netz voll).


In [ ]:
# ── Datensatz 2: CH Netzlast via ENTSO-E ────────────────────────────────────
# Gleicher API-Key wie für Preise.
# query_load('CH') liefert die stündliche Systemlast des Schweizer Regelblocks [MW].
# Jahresweiser Download mit Retry bei 503 — identisch zur DS1-Logik.

LOAD_FILE = os.path.join(DIR_RAW, 'ch_netzlast_raw.csv')


if not needs_download(LOAD_FILE, 10, 'netzlast'):
    print(f'Lastdaten vorhanden ({os.path.getsize(LOAD_FILE)/1024:.0f} KB) – kein Re-Download.')
    df_load = pd.read_csv(LOAD_FILE)

elif ENTSOE_API_KEY and portal_ok:
    print(f'Lade CH Netzlast {START_YEAR}–{END_YEAR} ({len(YEARS)} Jahre, jahresweise, max. 3 Retries bei 503)...')
    try:
        from entsoe import EntsoePandasClient
        client = EntsoePandasClient(api_key=ENTSOE_API_KEY)
        frames = []
        for year in YEARS:
            print(f'  Lade {year}...', end=' ')
            ts_year = fetch_entsoe_yearly(
                lambda s, e: client.query_load('CH', start=s, end=e),
                year)

            # query_load gibt je nach entsoe-py Version DataFrame oder Series zurück
            if isinstance(ts_year, pd.DataFrame):
                load_col = ts_year.columns[0]
                df_year  = ts_year[[load_col]].reset_index()
                df_year.columns = ['timestamp', 'load_mw']
            else:
                df_year = ts_year.to_frame(name='load_mw').reset_index()
                df_year.columns = ['timestamp', 'load_mw']

            frames.append(df_year)
            print(f'{len(df_year):,} h OK')

        df_load = pd.concat(frames, ignore_index=True)
        df_load = df_load[~df_load['timestamp'].duplicated(keep='first')]  # Überlapp-Schutz
        df_load['timestamp'] = pd.to_datetime(df_load['timestamp'], utc=True)
        df_load['load_gw']   = (df_load['load_mw'] / 1000).round(4)
        df_load = df_load[['timestamp', 'load_gw']].sort_values('timestamp').reset_index(drop=True)

        df_load.to_csv(LOAD_FILE, index=False, encoding='utf-8')
        kb = os.path.getsize(LOAD_FILE) / 1024
        log_dataindex(
            'ch_netzlast_raw.csv',
            'https://transparency.entsoe.eu (query_load CH)',
            LOAD_FILE, 'raw',
            rows=len(df_load), size_kb=kb,
            note=f'ENTSO-E query_load, jahresweise, MW→GW, {START_YEAR}–{END_YEAR}')
        print(f'Gespeichert: {LOAD_FILE} | {len(df_load):,} h | {kb:.0f} KB')

    except Exception as e:
        log_missing('ch_netzlast_raw.csv', str(e))
        raise RuntimeError(
            f'ENTSO-E query_load fehlgeschlagen: {e}\n'
            '→ NB01b_Daten_Sim ausführen (MODE=\'sim\' in ../sync/config.json setzen).')
else:
    reason = 'Kein API-Key' if not ENTSOE_API_KEY else 'Portal nicht erreichbar'
    log_missing('ch_netzlast_raw.csv', reason)
    raise RuntimeError(f'{reason} → NB01b_Daten_Sim ausführen.')

# Qualitätsprüfung: CH Systemlast typisch 5–12 GW
df_load_check = pd.read_csv(LOAD_FILE)
assert len(df_load_check) > 8000, f'Zu wenig Zeilen: {len(df_load_check)}'
assert df_load_check['load_gw'].between(2, 20).mean() > 0.98, 'Lastdaten ausserhalb plausiblem Bereich'
print(f'Qualität OK | Median: {df_load_check["load_gw"].median():.2f} GW | '
      f'{len(df_load_check):,} Zeilen')


**Verifikation DS2:** Shape, Zeitraum und Wertebereich der Netzlast prüfen.


In [ ]:
# ── Verifikation DS2: Netzlast ──────────────────────────────────────────────
print(f'Shape   : {df_load.shape}')
print(f'Zeitraum: {df_load["timestamp"].min()} – {df_load["timestamp"].max()}')
print(f'Nulls   : {df_load.isnull().sum().sum()}')
print(f'Range   : {df_load["load_gw"].min():.2f} – {df_load["load_gw"].max():.2f} GW')
df_load.head(3)


---
## Fazit <a id='fazit_NB_01'></a>

[↑ Inhaltsverzeichnis](#toc_NB_01)

Zwei vollständige Jahre stündlicher CH-Daten geladen (Spot-Preise und Netzlast),
Abdeckung gemäss ENTSO-E-Verfügbarkeit. Saubere Ausgangsbasis für die Bereinigung
in NB02 und die Wirtschaftlichkeitsrechnung in NB03.

Der Datenzeitraum (`start_date`, `end_date`, `n_years`) ist in
`../sync/transfer.json` hinterlegt und wird von allen Downstream-Notebooks
per Transfer-Lader eingelesen.


---
## Abschluss <a id='abschluss_NB_01'></a>

[↑ Inhaltsverzeichnis](#toc_NB_01)

Datenzeitraum in `../sync/transfer.json` schreiben (SSOT für NB02–K_99),
dann Ausgabedateien auf Existenz und Mindestgrösse prüfen.


**Transfer NB01 → downstream:** Schreibt `datenzeitraum` (Start/End/n_years)
in `../sync/transfer.json` — Single Source of Truth für alle nachfolgenden Notebooks.


**🔎 Quellcode der importierten lib-Funktion**

Die Funktion `save_transfer` wird aus `lib/io_ops.py` importiert und
in der folgenden Zelle verwendet. Aufklappbar ist der Quellcode einsehbar.


In [ ]:
show_source(save_transfer)


In [ ]:
# -- Transfer: Datenzeitraum in ../sync/transfer.json schreiben -----------------------
# Wird von NB02–K_99 gelesen; muss nach dem Laden der Preisdaten stehen,
# damit n_years aus echten Daten stammt.
_datenzeitraum = {
    'start_date': str(START_YEAR),
    'end_date':   str(END_YEAR),
    'n_years':    round(len(df_prices) / 8760, 3),
}
save_transfer(_datenzeitraum, key='datenzeitraum')
print(f"../sync/transfer.json: datenzeitraum {START_YEAR}–{END_YEAR} | "
      f"n_years={_datenzeitraum['n_years']:.3f}")


**Abschlusskontrolle:** Pflicht-Ausgabedateien auf Existenz und Mindestgrösse prüfen;
`../sync/dataindex.csv`-Status anzeigen. Fehler müssen vor NB02 behoben werden.


In [ ]:
# ── Abschlusskontrolle NB01 ──────────────────────────────────────────────────
print('NB01 – Abschlusskontrolle')
print('=' * 60)
for fname, min_kb, desc in [
    ('ch_spot_prices_raw.csv', 10, 'ENTSO-E Spot-Preise (DS1) — Pflicht'),
    ('ch_netzlast_raw.csv',    10, 'ENTSO-E Netzlast (DS2) — Pflicht'),
]:
    path = os.path.join(DIR_RAW, fname)
    if os.path.exists(path) and os.path.getsize(path)/1024 >= min_kb:
        kb = os.path.getsize(path)/1024
        tag = 'KB' if kb < 1024 else 'MB'
        val = kb if kb < 1024 else kb/1024
        print(f'  ✅  {fname:<40} {val:>7.1f} {tag}  ({desc})')
    else:
        print(f'  ❌  {fname:<40} FEHLT  ({desc})')

# Kür-Datensätze werden im ersten NB geladen, das sie benötigt:
#   Grenzflüsse → NB07_Cross_Border.ipynb (Sektion 1)
#   BFE GPKG    → NB06_Raeumliche_Analyse.ipynb (Sektion 1.1)
print()
print('Kür-Datensätze: werden in K_01 / K_02 bei Bedarf geladen.')

if os.path.exists('../sync/dataindex.csv'):
    df_idx = pd.read_csv('../sync/dataindex.csv')
    active = df_idx[df_idx['status']=='active']
    print(f'\ndataindex.csv: {len(df_idx)} Einträge total, {len(active)} active')
    print(active[['filename','data_type','rows','size_kb','timestamp']].to_string(index=False))

print('\n✅ Voraussetzungen für NB02 erfüllt' if True else '❌ Fehler beheben')


|  | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [NB02 Daten Bereinigung →](02_Daten_Bereinigung.ipynb) |
|:---|:---:|---:|
